# ✏️ Handwritten Digit Recognizer
An optimized CNN trained on MNIST with data augmentation, LR scheduling, and a live drawing app.

**Run everything at once:** Runtime → Run all (`Ctrl+F9`)

| Step | What you will do |
|------|------------------|
| 1 | Install & import libraries |
| 2 | Load the MNIST dataset |
| 3 | Explore & visualize the data |
| 4 | Preprocess |
| 5 | Build CNN with augmentation |
| 6 | Train with LR scheduling & early stopping |
| 7 | Evaluate, curves, confusion matrix |
| 7b | Visualize mistakes |
| 7c | Learning rate history |
| 8 | Save the model |
| 9 | Draw a digit live (Gradio) |

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow gradio seaborn pillow --quiet

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization,
    RandomRotation, RandomZoom, RandomTranslation
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

print('✅ All libraries imported!')
print(f'   TensorFlow: {tf.__version__}')

## Step 2 — Load the MNIST Dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print('📊 Dataset loaded!')
print(f'   Training images : {X_train.shape}  → {len(X_train)} images of 28×28 pixels')
print(f'   Test images     : {X_test.shape}   → {len(X_test)} images of 28×28 pixels')
print(f'   Labels          : digits 0 through {y_test.max()}')

## Step 3 — Explore & Visualize the Data

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    idx = np.random.randint(0, len(X_train))
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train[idx], cmap='gray')
    plt.title(f'Label: {y_train[idx]}', fontsize=10)
    plt.axis('off')
plt.suptitle('Sample MNIST Images', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
unique, counts = np.unique(y_train, return_counts=True)
plt.bar(unique, counts, color='darkslateblue', edgecolor='white')
plt.title('Training samples per digit')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.xticks(range(10))
plt.tight_layout()
plt.show()

print('💡 MNIST is well-balanced — ~6,000 samples per digit.')

## Step 4 — Preprocess the Data

In [ ]:
# Reshape: add channel dimension (CNNs expect H × W × C)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Normalize pixel values 0–255 → 0.0–1.0
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# One-hot encode labels: e.g. 3 → [0,0,0,1,0,0,0,0,0,0]
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat  = to_categorical(y_test,  num_classes=10)

print('✅ Preprocessing complete!')
print(f'   X_train shape : {X_train.shape}')
print(f'   y_train shape : {y_train_cat.shape}')
print(f'   Pixel range   : {X_train.min():.1f} → {X_train.max():.1f}')

## Step 5 — Build the CNN Model

In [ ]:
model = Sequential([
    # --- Data Augmentation (only active during training) ---
    RandomRotation(0.08,  fill_mode='constant', fill_value=0.0),
    RandomZoom(0.08,      fill_mode='constant', fill_value=0.0),
    RandomTranslation(0.08, 0.08, fill_mode='constant', fill_value=0.0),

    # --- Block 1: detect edges and basic shapes (32 filters) ---
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # --- Block 2: detect curves and digit parts (64 filters) ---
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # --- Block 3: detect full digit structure (128 filters) ---
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.25),

    # --- Classification head ---
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.build(input_shape=(None, 28, 28, 1))
model.summary()

## Step 6 — Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Halves the learning rate when val_loss stops improving for 2 epochs
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_train, y_train_cat,
    epochs=25,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f'\n✅ Training complete — stopped at epoch {len(history.history["loss"])}')

## Step 7 — Evaluate the Model

In [ ]:
loss, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Accuracy : {acc*100:.2f}%')
print(f'Test Loss     : {loss:.4f}\n')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred))

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## Step 7b — Visualize Mistakes

In [ ]:
wrong_idx = np.where(y_pred != y_test)[0]
print(f'Total mistakes: {len(wrong_idx)} / {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)')

plt.figure(figsize=(12, 8))
for i, idx in enumerate(wrong_idx[:16]):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f'True: {y_test[idx]}  Pred: {y_pred[idx]}', fontsize=9, color='crimson')
    plt.axis('off')
plt.suptitle('Images the model got wrong', fontsize=13)
plt.tight_layout()
plt.show()

## Step 7c — Learning Rate History
> Shows exactly when ReduceLROnPlateau fired and reduced the learning rate.

In [ ]:
lr_history = history.history.get('learning_rate', history.history.get('lr', []))

if lr_history:
    plt.figure(figsize=(8, 3))
    plt.plot(lr_history, marker='o', color='darkorange')
    plt.title('Learning Rate over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

    # Find epochs where LR dropped
    drops = [i for i in range(1, len(lr_history)) if lr_history[i] < lr_history[i-1]]
    if drops:
        print(f'LR was reduced at epoch(s): {[d+1 for d in drops]}')
    else:
        print('LR was never reduced — training converged smoothly!')
else:
    print('LR history not recorded.')

## Step 8 — Save the Model

In [ ]:
model.save('digit_recognizer.keras')
print('✅ Model saved as digit_recognizer.keras')

# To reload later:
# model = load_model('digit_recognizer.keras')

## Step 9 — Draw a Digit Live (Gradio)

In [ ]:
import gradio as gr

def predict_digit(img):
    if img is None:
        return {str(i): 0.0 for i in range(10)}

    # Sketchpad returns a dict in newer Gradio — extract the image array
    if isinstance(img, dict):
        img = img.get('composite', img.get('background', None))
    if img is None:
        return {str(i): 0.0 for i in range(10)}

    # Convert to PIL, resize to 28x28 grayscale
    pil_img = Image.fromarray(img.astype('uint8'))
    pil_img = pil_img.convert('L')           # grayscale
    pil_img = pil_img.resize((28, 28), Image.LANCZOS)

    arr = np.array(pil_img).astype('float32')

    # IMPORTANT: MNIST is white digits on black background.
    # Sketchpad draws black strokes on white — so we invert.
    arr = 255.0 - arr

    arr = arr / 255.0
    arr = arr.reshape(1, 28, 28, 1)

    probs = model.predict(arr, verbose=0)[0]
    return {str(i): float(probs[i]) for i in range(10)}

gr.Interface(
    fn=predict_digit,
    inputs=gr.Sketchpad(
        label='Draw any digit (0–9)',
        type='numpy'
    ),
    outputs=gr.Label(num_top_classes=3, label='Top Predictions'),
    title='✏️ Handwritten Digit Recognizer',
    description='Draw a digit clearly in the box. The model will show its top 3 guesses.'
).launch(debug=False)